# Cross Collapse Operator

## Variables
+ Two random variables $X$, $Y$
+ conditional means $\mu^j_X = E[X \mid S=j]$, $\mu^j_Y = E[Y \mid S=j]$
+ conditional cross variance $V^j_{X, Y} = Cov[X, Y \mid S=j]$
+ mixing coefficients $P^j = Pr(S = j)$


Want to compute the unconditional mean ($\mu_X$ and $\mu_Y$) and covariance ($V_{X,Y}$) of $X$ and $Y$.

## Definitions
$$
\mu_X = \sum_j P^j \mu^j_X
$$
$$
\mu_Y = \sum_j P^j \mu^j_Y
$$

$$
\begin{equation}
\begin{split}
V_{X,Y} & = \sum_j P^j V^j_{X, Y} \\
 & = \sum_j P^j Cov[X, Y \mid S=j] \\
 & = \sum_j P^j E[(X - \mu_X)(Y - \mu_Y)^T \mid S = j] \\
 & = \sum_j P^j E[(X - \mu^j_X + \mu^j_X - \mu_X)(Y - \mu^j_Y + \mu^j_Y - \mu_Y)^T \mid S = j] \\
 & = \sum_j P^j E[(X - \mu^j_X)(Y - \mu_Y)^T \mid S = j]  + \sum_j P^j(\mu^j_X - \mu_X)(\mu^j_Y - \mu_Y)^T\\
 & = \sum_j P^j V^j_{X, Y} + \sum_j P^j(\mu^j_X - \mu_X)(\mu^j_Y - \mu_Y)^T\\
\end{split}
\end{equation}
$$

In [ ]:
import numpy as np

n_states = 2
n_random_variable_dim = 1

conditional_mean_x = np.arange(n_states)[
    None
]  # shape (n_random_variable_dim, n_states,)
conditional_mean_y = np.arange(n_states)[
    None
]  # shape (n_random_variable_dim, n_states,)
mixing_coef = np.ones((n_states,)) / n_states  # shape (n_states,)
# mixing_coef = np.array([0.1, 0.9])  # shape (n_states,)
conditional_cross_variance = np.stack(
    [np.eye(n_random_variable_dim) * 0.9 for _ in range(n_states)], axis=-1
)
# shape (n_random_variable_dim, n_random_variable_dim, n_states)

In [ ]:
mean_x = conditional_mean_x @ mixing_coef  # shape (n_random_variable_dim,)
mean_y = conditional_mean_y @ mixing_coef  # shape (n_random_variable_dim,)

# cross variance shape (n_random_variable_dim, n_random_variable_dim)
cross_variance = np.zeros((n_random_variable_dim, n_random_variable_dim))

ft = []
st = []
for i in range(n_states):
    first_term = mixing_coef[i] * conditional_cross_variance[..., i]
    ft.append(first_term)
    second_term = mixing_coef[i] * np.outer(
        conditional_mean_x[..., i] - mean_x, conditional_mean_y[..., i] - mean_y
    )
    st.append(second_term)
    cross_variance += first_term + second_term

cross_variance

array([[1.15]])

In [ ]:
diff_x = conditional_mean_x - mean_x[:, None]  # shape (n_random_variable_dim, n_states)
diff_y = conditional_mean_y - mean_y[:, None]  # shape (n_random_variable_dim, n_states)

first_term = (
    conditional_cross_variance @ mixing_coef
)  # shape (n_random_variable_dim, n_random_variable_dim)
second_term = np.einsum(
    "ij,jk->ik",
    diff_x * mixing_coef,
    diff_y.T,
)  # shape (n_random_variable_dim, n_random_variable_dim)
second_term

array([[0.25]])

In [45]:
(diff_x * mixing_coef) @ diff_y.T

array([[0.25]])

In [26]:
conditional_mean_x

array([[0, 1]])

In [ ]:
import jax.numpy as jnp


from state_space_practice.switching_kalman import (
    collapse_gaussian_mixture_cross_covariance,
)


def test_collapse_shapes():
    """
    Test that the output shapes are correct for a given input shape.
    """
    # Suppose we have 2 dimensions (X, Y each has dimension 2)
    # and 3 discrete states.
    n_dims = 2
    n_states = 3

    conditional_means_x = jnp.zeros((n_dims, n_states))
    conditional_means_y = jnp.zeros((n_dims, n_states))
    conditional_cross_cov = jnp.zeros((n_dims, n_dims, n_states))
    mixing_weights = jnp.ones((n_states,)) / n_states

    mean_x, mean_y, cov_xy = collapse_gaussian_mixture_cross_covariance(
        conditional_means_x, conditional_means_y, conditional_cross_cov, mixing_weights
    )

    # mean_x, mean_y should be shape (n_dims,)
    assert mean_x.shape == (n_dims,)
    assert mean_y.shape == (n_dims,)

    # cov_xy should be shape (n_dims, n_dims)
    assert cov_xy.shape == (n_dims, n_dims)


def test_collapse_single_component():
    """
    Test that if there is only one mixture component (p=1.0),
    the result is exactly that component's mean/cov.
    """
    n_dims = 2
    n_states = 1

    # Single-state means/cov
    mu_x = jnp.array([[1.0], [2.0]])  # shape (2,1)
    mu_y = jnp.array([[3.0], [4.0]])  # shape (2,1)
    cross_cov = jnp.array([[[10.0], [1.0]], [[1.0], [20.0]]])  # shape (2,2,1)

    weights = jnp.array([1.0])  # single mixture component

    mean_x, mean_y, cov_xy = collapse_gaussian_mixture_cross_covariance(
        mu_x, mu_y, cross_cov, weights
    )

    # They should match exactly
    assert jnp.allclose(mean_x, mu_x.squeeze(axis=1))
    assert jnp.allclose(mean_y, mu_y.squeeze(axis=1))
    # The cross-cov for a single component is just that component's cross-cov
    assert jnp.allclose(cov_xy, cross_cov.squeeze(axis=2))


def test_collapse_equal_weights():
    """
    Test that for two components with equal weights, the means/covariances
    are exactly the average of the two components.
    """
    n_dims = 2
    n_states = 2

    # Two mixture components
    mu_x = jnp.array([[1.0, 3.0], [2.0, 4.0]])  # shape (2,2)
    mu_y = jnp.array([[5.0, 7.0], [6.0, 8.0]])  # shape (2,2)
    cross_cov = jnp.array(
        [[[10.0, 30.0], [1.0, 99.0]], [[1.0, 99.0], [20.0, 40.0]]]
    )  # shape (2,2,2)

    weights = jnp.array([0.5, 0.5])

    # Compute collapsed distribution
    mean_x, mean_y, cov_xy = collapse_gaussian_mixture_cross_covariance(
        mu_x, mu_y, cross_cov, weights
    )

    # Manually compute expected means
    expected_mean_x = (mu_x[:, 0] + mu_x[:, 1]) * 0.5
    expected_mean_y = (mu_y[:, 0] + mu_y[:, 1]) * 0.5

    # Manually compute expected cross-cov
    # sum_j p_j Cov_j + sum_j p_j (mu^j_x - mean_x)(mu^j_y - mean_y)^T
    # We'll do it directly:
    cov_part = (cross_cov[:, :, 0] * 0.5) + (cross_cov[:, :, 1] * 0.5)

    diff_x0 = (mu_x[:, 0] - expected_mean_x).reshape(-1, 1)
    diff_x1 = (mu_x[:, 1] - expected_mean_x).reshape(-1, 1)
    diff_y0 = (mu_y[:, 0] - expected_mean_y).reshape(1, -1)
    diff_y1 = (mu_y[:, 1] - expected_mean_y).reshape(1, -1)

    outer_0 = 0.5 * diff_x0 @ diff_y0
    outer_1 = 0.5 * diff_x1 @ diff_y1

    expected_cov_xy = cov_part + outer_0 + outer_1

    assert jnp.allclose(mean_x, expected_mean_x)
    assert jnp.allclose(mean_y, expected_mean_y)
    assert jnp.allclose(cov_xy, expected_cov_xy, atol=1e-7)


test_collapse_equal_weights()
test_collapse_shapes()
test_collapse_single_component()

In [54]:
np.sqrt(12)

3.4641016151377544

In [ ]:
# smoother_gain = state_covariance_t @ transition_matrix.T @ state_covariance_{t+1}^{-1}
# smoother_mean_t = filter_mean_t + smoother_gain @ (smoother_mean_{t+1} - filter_mean_{t+1})
# smoother_covariance_t = filter_covariance_t + smoother_gain @ (smoother_covariance_{t+1} - filter_covariance_{t+1}) @ smoother_gain.T
# smoother_cross_covariance_t = filter_covariance_t @ smoother_gain_{t-1}^T + smoother_gain @ (smoother_cross_covariance_{t+1} - transition_matrix @ filter_covariance_t) @ smoother_gain.T

In [ ]:
def outer_product(x, y):
    """
    Computes the outer product of two vectors.

    Parameters
    ----------
    x : np.ndarray, shape (n_time, n_random_variable_dim)
        First vector.
    y : np.ndarray, shape (n_time, n_random_variable_dim)
        Second vector.

    Returns
    -------
    np.ndarray, shape (n_random_variable_dim, n_random_variable_dim)
        Outer product of x and y.
    """
    return np.einsum("ti,tj->ij", x, y)


def outer_product_loop(x, y):
    """
    Computes the outer product of two vectors using a loop.
    """
    result = np.zeros((x.shape[1], y.shape[1]))
    for t in range(x.shape[0]):
        result[t] += np.outer(x[t], y[t])
    return result


x = np.array([[1, 2], [3, 4], [5, 6]])
y = np.array([[7, 8], [9, 10], [11, 12]])
outer_product(x, y)

array([[ 89,  98],
       [116, 128]])

In [71]:
x.T @ y

array([[ 89,  98],
       [116, 128]])

In [ ]:
np.sum(np.einsum("ti,tj->tij", x, y), axis=0)

array([[ 89,  98],
       [116, 128]])

In [60]:
x.shape, y.shape

((3, 2), (3, 2))

In [ ]:
(x[[0]] @ y[[0]].T).shape

(1, 1)

In [106]:
import jax


def weighted_outer_sum(x: jnp.ndarray, y: jnp.ndarray, weights: jnp.ndarray):
    """Compute the weighted outer sum of two arrays.
    Parameters
    ----------
    x : jnp.ndarray, shape (n_time, n_dims)
        First array.
    y : jnp.ndarray, shape (n_time, n_dims)
        Second array.
    weights : jnp.ndarray, shape (n_time,)
        Weights for the outer sum.

    Returns
    -------
    weighted_outer_sum: jnp.ndarray, shape (n_dims, n_dims)
        Weighted outer sum of x and y.
    """
    return (x * weights).T @ y


def weighted_outer_sum2(x, y, weights):
    return jnp.stack(
        [(x[..., i] * weights[:, [i]]).T @ y[..., i] for i in range(weights.shape[1])],
        axis=-1,
    )


def weighted_outer_sum3(x, y, weights):
    return jnp.einsum("txd, tyd, td -> xyd", x, y, weights)


n_time = 10
n_cont_states = 2
n_discrete_states = 3
smoother_means = np.random.uniform(size=(n_time, n_cont_states, n_discrete_states))
smoother_discrete_state_prob = np.random.uniform(size=(n_time, n_discrete_states))
smoother_covs = np.random.uniform(size=(n_time, n_cont_states, n_cont_states, n_discrete_states))

jnp.allclose(
    weighted_outer_sum3(smoother_means, smoother_means, smoother_discrete_state_prob),
    weighted_outer_sum2(smoother_means, smoother_means, smoother_discrete_state_prob),
)

Array(True, dtype=bool)

In [99]:
smoother_discrete_state_prob[-1]

array([0.99033895, 0.21689698, 0.6630782 ])

In [100]:
smoother_means[-1].T @ smoother_means[-1]

array([[0.88845905, 0.75610769, 0.12461529],
       [0.75610769, 0.88243505, 0.3811629 ],
       [0.12461529, 0.3811629 , 0.33420656]])

In [111]:
blah = jnp.stack(
    [
        smoother_discrete_state_prob[-1, i]
        * jnp.outer(smoother_means[-1, ..., i], smoother_means[-1, ..., i])
        for i in range(n_discrete_states)
    ], axis=-1
)
blah.shape

(2, 2, 3)

In [116]:
blah2 = jnp.einsum("cm, dm, m -> cdm", smoother_means[-1], smoother_means[-1], smoother_discrete_state_prob[-1])
jnp.allclose(blah, blah2)

Array(True, dtype=bool)

In [110]:
(smoother_covs[-1] * smoother_discrete_state_prob[-1, None, None]).shape

(2, 2, 3)

In [ ]:
smoother_cross_cov = np.random.uniform(size=(n_time - 1, n_cont_states, n_cont_states, n_discrete_states))
jnp.swapaxes(smoother_cross_cov * smoother_discrete_state_prob[1:, None, None], 1, 2).sum(axis=0)

(2, 2, 3)

In [125]:
def weighted_outer_sum(x: jnp.ndarray, y: jnp.ndarray, weights: jnp.ndarray):
    """Compute the weighted outer sum of two arrays.
    Parameters
    ----------
    x : jnp.ndarray, shape (n_time, x_dims, n_discrete_states)
        First array.
    y : jnp.ndarray, shape (n_time, y_dims, n_discrete_states)
        Second array.
    weights : jnp.ndarray, shape (n_time, n_discrete_states)
        Weights for the outer sum.

    Returns
    -------
    weighted_outer_sum: jnp.ndarray, shape (x_dims, y_dims, n_discrete_states)
        Weighted outer sum of x and y.
    """
    return jnp.einsum("tcm, tdm, tm -> cdm", x, y, weights)

n_obs_dim = 4
obs = np.random.uniform(size=(n_time, n_obs_dim))
alpha = weighted_outer_sum(obs[..., None], obs[..., None], smoother_discrete_state_prob)
alpha.shape

(4, 4, 3)

In [127]:
(obs.T @ obs).shape

(4, 4)

In [129]:
weighted_outer_sum(smoother_means[[0]], smoother_means[[0]], smoother_discrete_state_prob[[0]]).shape

(2, 2, 3)

In [132]:
jnp.allclose(
    weighted_outer_sum(
        smoother_means[[-1]], smoother_means[[-1]], smoother_discrete_state_prob[[-1]]
    ),
    jnp.einsum(
        "cm, dm, m -> cdm",
        smoother_means[-1],
        smoother_means[-1],
        smoother_discrete_state_prob[-1],
    ),
)

Array(True, dtype=bool)

In [135]:
n_time = smoother_discrete_state_prob.sum(axis=0)
n_time_1 = smoother_discrete_state_prob[1:].sum(axis=0)

# Compute intermediate expectation terms
gamma = jnp.sum(
    smoother_covs * smoother_discrete_state_prob[:, None, None], axis=0
) + weighted_outer_sum(smoother_means, smoother_means, smoother_discrete_state_prob)
delta = weighted_outer_sum(
    obs[..., None], smoother_means, smoother_discrete_state_prob
)
alpha = weighted_outer_sum(
    obs[..., None], obs[..., None], smoother_discrete_state_prob
)

last_gamma = (
    smoother_covs[-1] * smoother_discrete_state_prob[-1, None, None]
) + weighted_outer_sum(
    smoother_means[[-1]], smoother_means[[-1]], smoother_discrete_state_prob[[-1]]
)
first_gamma = (
    smoother_covs[0] * smoother_discrete_state_prob[0, None, None]
) + weighted_outer_sum(
    smoother_means[[0]], smoother_means[[0]], smoother_discrete_state_prob[[0]]
)
gamma1 = gamma - last_gamma
gamma2 = gamma - first_gamma

beta = jnp.swapaxes(
    smoother_cross_cov * smoother_discrete_state_prob[1:, None, None], 1, 2
).sum(axis=0)

In [139]:
gamma.shape # (n_cont_states, n_cont_states, n_discrete_states)
beta.shape # (n_cont_states, n_cont_states, n_discrete_states)
delta.shape # (n_obs_dim, n_cont_states, n_discrete_states)
alpha.shape # (n_obs_dim, n_obs_dim, n_discrete_states)

(4, 4, 3)

In [ ]:
measurement_matrix = np.random.uniform(
    size=(n_obs_dim, n_cont_states, n_discrete_states)
)

(measurement_matrix @ jnp.swapaxes(delta, 0, 1)).shape

ValueError: Incompatible shapes for matmul arguments: (4, 2, 3) and (2, 4, 3)

In [146]:
jnp.swapaxes(delta, 1, 2).shape

(4, 3, 2)

In [150]:
def symmetrize(A):
    """Symmetrize one or more matrices."""
    return 0.5 * (A + jnp.swapaxes(A, -1, -2))


def psd_solve(A, b, diagonal_boost=1e-9):
    """A wrapper for coordinating the linalg solvers used in the library for psd matrices."""
    return jax.scipy.linalg.solve(
        symmetrize(A) + diagonal_boost * jnp.eye(A.shape[-1]), b, assume_a="pos"
    )


psd_solve_per_discrete_state = jax.vmap(
    lambda x, y: psd_solve(x, y.T).T, in_axes=(-1, -1), out_axes=-1
)

measurement_matrix = psd_solve_per_discrete_state(gamma, delta)

In [151]:
measurement_matrix.shape

(4, 2, 3)

In [ ]:
measurement_cov = (alpha - measurement_matrix @ delta.T) / n_time
measurement_cov = symmetrize(measurement_cov)

(4, 2, 3)

In [155]:
psd_solve_per_discrete_state = jax.vmap(
    lambda x, y: psd_solve(x, y.T).T, in_axes=(-1, -1), out_axes=-1
)

cov_solve_per_discrete_state = jax.vmap(
    lambda x, y, z, n: symmetrize((x - y @ z.T) / n),
    in_axes=(-1, -1, -1, -1),
    out_axes=-1,
)

In [161]:
smoother_joint_discrete_state_prob = jnp.zeros(
    (10, n_discrete_states, n_discrete_states)
)

In [162]:
n_time = smoother_discrete_state_prob.sum(axis=0)
n_time_1 = smoother_discrete_state_prob[1:].sum(axis=0)

# Compute intermediate expectation terms
gamma = jnp.sum(
    smoother_covs * smoother_discrete_state_prob[:, None, None], axis=0
) + weighted_outer_sum(smoother_means, smoother_means, smoother_discrete_state_prob)

delta = weighted_outer_sum(
    obs[..., None], smoother_means, smoother_discrete_state_prob
)
alpha = weighted_outer_sum(
    obs[..., None], obs[..., None], smoother_discrete_state_prob
)

last_gamma = (
    smoother_covs[-1] * smoother_discrete_state_prob[-1, None, None]
) + weighted_outer_sum(
    smoother_means[[-1]], smoother_means[[-1]], smoother_discrete_state_prob[[-1]]
)
first_gamma = (
    smoother_covs[0] * smoother_discrete_state_prob[0, None, None]
) + weighted_outer_sum(
    smoother_means[[0]], smoother_means[[0]], smoother_discrete_state_prob[[0]]
)
gamma1 = gamma - last_gamma
gamma2 = gamma - first_gamma

beta = jnp.swapaxes(
    smoother_cross_cov * smoother_discrete_state_prob[1:, None, None], 1, 2
).sum(axis=0)

# gamma: (n_cont_states, n_cont_states, n_discrete_states)
# beta: (n_cont_states, n_cont_states, n_discrete_states)
# alpha: (n_obs_dim, n_obs_dim, n_discrete_states)
# delta: (n_obs_dim, n_cont_states, n_discrete_states)

# Measurement matrix and covariance
measurement_matrix = psd_solve_per_discrete_state(gamma, delta)
# measurement_matrix: shape (n_obs_dim, n_cont_states, n_discrete_states)
measurement_cov = cov_solve_per_discrete_state(
    alpha, measurement_matrix, delta, n_time
)

# Transition matrix
continuous_transition_matrix = psd_solve_per_discrete_state(gamma1, beta)

# Process covariance
process_cov = cov_solve_per_discrete_state(
    gamma2, continuous_transition_matrix, beta, n_time_1
)
# Initial mean and covariance
init_mean = smoother_means[0]
init_cov = smoother_covs[0]

# Discrete transition matrix
discrete_state_transition = smoother_joint_discrete_state_prob.sum(
    axis=0
) / smoother_discrete_state_prob[:-1].sum(axis=0)

init_discrete_state_prob = smoother_discrete_state_prob[0]